# Joint action + value pretrain (one shared L2)

Trains the **action head** (PairHead, single-target contract) and the
**value heads** (win / rank / score / current-state / momentum / near-future
is_ahead) **together**, both branching from the **same L2** of
`EntityPretrainModel` — action via L3/L4/PairHead, value via the
PlayerConsolidator. **L0+L1 frozen, L2 and up trainable** (`freeze_below_l2`).

- **Action labels** = pair cache (`--pair-cache-path`).
- **Value labels** = cross-entity cache (`--cross-cache-path`).
- **Near-future decay**: the future is_ahead reward over horizons {10,20,50}
  is weighted by `gamma**K` (`--fwd-gamma`, default 0.9 → K=10 ~73%).
- **Verbose per-head logging** each epoch (action + value groups).

The shared L2 is built at `n_steps=10`, so it consumes both the T=6 pair
cache and the T=10 cross-entity cache via `step_embed[-T:]`.


## 1. Authenticate + pull bundle from GCS

In [ ]:
from google.colab import auth
auth.authenticate_user()
BUCKET = 'gs://orbit-wars-shipping/entity'
# Action cache (pair). 'pair_cache' = bow+Ebi ~13 GB; 'pair_cache_top4' ~33 GB.
PAIR_CACHE_PREFIX = 'pair_cache'
# Value cache (cross-entity), chunk+uploaded by scripts/chunk_cross_entity_cache.sh.
CROSS_CACHE_PREFIX = 'cross_entity_cache_joint'
print(f'pulling from {BUCKET}\n  action={PAIR_CACHE_PREFIX}  value={CROSS_CACHE_PREFIX}')


In [ ]:
import os, subprocess, time, hashlib, json, concurrent.futures
from pathlib import Path

WORK = Path('/content/orbit-wars'); WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

def _gcs_size(url):
    try:
        out = subprocess.run(['gcloud','storage','objects','describe',url,
                              '--format=value(size)'], check=True,
                             capture_output=True, text=True)
        return int(out.stdout.strip())
    except Exception:
        return None

def _cp(src, dst, force=True):
    dst = Path(dst)
    if dst.exists() and not force:
        return dst.stat().st_size
    if dst.exists():
        dst.unlink()
    print(f'  pulling {src} -> {dst.name} ...', flush=True)
    subprocess.run(['gcloud','storage','cp',src,str(dst)], check=True)
    return dst.stat().st_size

def _sha256(p):
    h = hashlib.sha256()
    with open(p,'rb') as fh:
        for blk in iter(lambda: fh.read(1<<20), b''): h.update(blk)
    return h.hexdigest()

def pull_cache(prefix, dst):
    """Chunked (manifest) or single-object cache pull + assemble."""
    dst = Path(dst)
    man_url = f'{BUCKET}/{prefix}.manifest.json'
    if _gcs_size(man_url) is not None:
        man = json.loads(subprocess.run(['gcloud','storage','cat',man_url],
                         check=True, capture_output=True, text=True).stdout)
        total = int(man.get('total_bytes', 0))
        if dst.exists() and total and dst.stat().st_size == total:
            print(f'  {dst.name}: cached ({total/1024**3:.2f} GB)'); return
        cdir = WORK / f'{prefix}_chunks'; cdir.mkdir(exist_ok=True)
        def _pull(spec):
            cp = cdir / spec['name']
            if not (cp.exists() and cp.stat().st_size == int(spec.get('size_bytes',0))):
                subprocess.run(['gcloud','storage','cp',f"{BUCKET}/{spec['name']}",str(cp)], check=True)
            if spec.get('sha256') and _sha256(cp) != spec['sha256']:
                raise RuntimeError(f"sha256 mismatch {spec['name']}")
            return spec['name'], cp.stat().st_size
        print(f'  {prefix}: {len(man["chunks"])} chunks, {total/1024**3:.2f} GB')
        with concurrent.futures.ThreadPoolExecutor(max_workers=len(man['chunks'])) as pool:
            for nm,sz in pool.map(_pull, man['chunks']):
                print(f'    {nm}  {sz/1024**2:.1f} MB')
        if dst.exists(): dst.unlink()
        with open(dst,'wb') as out:
            for c in man['chunks']:
                with open(cdir/c['name'],'rb') as fh:
                    while True:
                        b = fh.read(1<<22)
                        if not b: break
                        out.write(b)
        print(f'  assembled {dst.name}: {dst.stat().st_size/1024**3:.2f} GB')
        return
    for cand in (f'{prefix}.pt', f'{prefix}'):
        if _gcs_size(f'{BUCKET}/{cand}') is not None:
            sz = _cp(f'{BUCKET}/{cand}', dst, force=not dst.exists())
            print(f'  {dst.name}: {sz/1024**3:.2f} GB (single)'); return
    raise RuntimeError(f'no cache for prefix {prefix} on {BUCKET}')

t0 = time.time()
_cp(f'{BUCKET}/code.tgz', WORK/'code.tgz')
_cp(f'{BUCKET}/weights.tgz', WORK/'weights.tgz')
pull_cache(PAIR_CACHE_PREFIX, WORK/'pair_cache.pt')
pull_cache(CROSS_CACHE_PREFIX, WORK/'cross_entity_cache.pt')
print(f'pull done in {time.time()-t0:.1f}s')


In [ ]:
# Wipe stale extracted code; leave the big caches alone.
!rm -rf agents scripts ckpts
!find . -maxdepth 1 -name '*.pt' ! -name 'pair_cache.pt' ! -name 'cross_entity_cache.pt' -delete
!tar xzf code.tgz
!tar xzf weights.tgz
import sys, importlib, gc
for m in list(sys.modules):
    if m.startswith('agents') or m.startswith('scripts'): del sys.modules[m]
importlib.invalidate_caches(); gc.collect()
!find . -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null || true
print('extracted code.tgz + weights.tgz')


## 2. Verify the joint model wiring (value heads off shared L2)

In [ ]:
import torch, agents
from agents.transformer_v2.pretrain.entity_encoder import EntityPretrainModel
from agents.transformer_v2.pretrain.value_heads import ValuePretrainHeads
from agents.transformer_v2.pretrain import joint_pretrain
assert 'Minimal' in (agents.__doc__ or ''), 'stale agents shim — restart kernel'
m = EntityPretrainModel(d_model=256, n_steps=10, with_consolidator=True, with_value_heads=True)
for a in ('entity','cross','dual_role','joint_role','pair_head','consolidator','value_heads'):
    assert getattr(m, a) is not None, f'missing {a} (stale code.tgz)'
assert isinstance(m.value_heads, ValuePretrainHeads)
rep = m.freeze_below_l2()
assert not any(p.requires_grad for p in m.entity.parameters()), 'L1 must be frozen'
assert all(p.requires_grad for p in m.cross.parameters()), 'L2 must be trainable'
assert all(p.requires_grad for p in m.value_heads.parameters()), 'value heads must train'
assert hasattr(joint_pretrain, 'train_joint')
n_tr = sum(p.numel() for p in m.parameters() if p.requires_grad)
print('joint model OK — value heads on shared L2; L1 frozen, L2+ trainable')
print(f'trainable params (L2+): {n_tr:,}')
for k,v in rep.items(): print(f'  {k:<28s} {v:,}')


## 3. GPU check

In [ ]:
import torch
print('cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(cpu)')


## 4. Stage frozen L0 encoders into run-dir layout

In [ ]:
import shutil
from pathlib import Path
PLANET_RUN_DIR = Path('/content/orbit-wars/ckpts/planet')
FLEET_RUN_DIR  = Path('/content/orbit-wars/ckpts/fleet')
COMET_RUN_DIR  = Path('/content/orbit-wars/ckpts/comet')
for d in (PLANET_RUN_DIR, FLEET_RUN_DIR, COMET_RUN_DIR): d.mkdir(parents=True, exist_ok=True)
shutil.copy('/content/orbit-wars/planet_encoder_best.pt', PLANET_RUN_DIR/'planet_encoder_best.pt')
shutil.copy('/content/orbit-wars/fleet_encoder_best.pt',  FLEET_RUN_DIR /'fleet_encoder_best.pt')
shutil.copy('/content/orbit-wars/comet_past_best.pt',     COMET_RUN_DIR /'comet_past_best.pt')
import torch
pc = torch.load(PLANET_RUN_DIR/'planet_encoder_best.pt', map_location='cpu', weights_only=False)
fc = torch.load(FLEET_RUN_DIR /'fleet_encoder_best.pt',  map_location='cpu', weights_only=False)
cc = torch.load(COMET_RUN_DIR /'comet_past_best.pt',     map_location='cpu', weights_only=False)
assert pc['config']['d_model']==fc['config']['d_model']==cc['config']['d_model']==256
print('L0 staged, all d=256')


## 5. Cache paths (the joint CLI reads these .pt directly)

In [ ]:
PAIR_CACHE_PATH  = '/content/orbit-wars/pair_cache.pt'
CROSS_CACHE_PATH = '/content/orbit-wars/cross_entity_cache.pt'
import os
for p in (PAIR_CACHE_PATH, CROSS_CACHE_PATH):
    assert os.path.exists(p), p
    print(f'{p}  {os.path.getsize(p)/1024**3:.2f} GB')


## 6. Train (joint action + value, L2~ unfreeze)

In [ ]:
import time
TS = time.strftime('%Y%m%d-%H%M%S')
RUN_TAG = f'joint_actval_d256_T10_{TS}'
OUT_DIR = f'data/runs/joint/{RUN_TAG}'
# hyperparams
D_MODEL, N_STEPS   = 256, 10
BATCH_SIZE, EPOCHS = 16, 20
LR, WEIGHT_DECAY   = 1e-4, 1e-4
LAUNCH_WEIGHT      = 4.0   # up-weight launches vs NOOP rows (single-target)
FWD_GAMMA          = 0.9   # large near-future decay on the is_ahead reward
VALUE_COEF         = 1.0   # value-loss weight relative to action loss
print('RUN_TAG =', RUN_TAG)


In [ ]:
import subprocess
cmd = [
    'python','-u','-m','agents.transformer_v2.pretrain.joint_pretrain',
    '--out-dir', OUT_DIR,
    '--fleet-run-dir', str(FLEET_RUN_DIR),
    '--planet-run-dir', str(PLANET_RUN_DIR),
    '--comet-run-dir', str(COMET_RUN_DIR),
    '--pair-cache-path', PAIR_CACHE_PATH,
    '--cross-cache-path', CROSS_CACHE_PATH,
    '--d-model', str(D_MODEL), '--n-steps', str(N_STEPS),
    '--batch-size', str(BATCH_SIZE), '--epochs', str(EPOCHS),
    '--lr', str(LR), '--weight-decay', str(WEIGHT_DECAY),
    '--launch-weight', str(LAUNCH_WEIGHT),
    '--fwd-gamma', str(FWD_GAMMA), '--value-coef', str(VALUE_COEF),
    '--device', 'cuda', '--progress-every', '50',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


## 7. Push the trained run back to GCS

In [ ]:
import subprocess
from pathlib import Path
src = Path(OUT_DIR); assert src.is_dir(), src
subprocess.run(['gcloud','storage','cp','--recursive',str(src),f'{BUCKET}/runs/'], check=True)
print('uploaded to', f'{BUCKET}/runs/{src.name}/')
subprocess.run(['gcloud','storage','ls','--long','--readable-sizes',
                f'{BUCKET}/runs/{src.name}/'], check=False)
